In [ ]:
import numpy as np
from keras.datasets import imdb
from collections import Counter
import math

np.random.seed(42)

vocab_size = 10000
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=vocab_size)

vocab = list(range(vocab_size))

concatenated = []
for review in X_train:
    for word in review:
        concatenated.append(word)
concatenated = np.array(concatenated)

alpha = 0.05
iterations = 1
hidden_size = 50
window = 2
negative = 5

weights_0_1 = np.random.rand(vocab_size, hidden_size) - 0.5
weights_1_2 = np.random.rand(vocab_size, hidden_size) - 0.5

layer_2_target = np.zeros(negative + 1)
layer_2_target[0] = 1

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

for iteration in range(iterations):
    for review in X_train:
        for target_i in range(len(review)):
            target_word = review[target_i]
            start = max(0, target_i - window)
            end = min(len(review), target_i + window + 1)
            context_words = review[start:target_i] + review[target_i+1:end]
            for context_word in context_words:
                random_words = list(np.random.randint(0, vocab_size, negative))
                targets = [context_word] + random_words
                layer_1 = weights_0_1[target_word]
                layer_2 = sigmoid(np.dot(layer_1, weights_1_2[targets].T))
                layer_2_delta = layer_2 - layer_2_target
                layer_1_delta = layer_2_delta.dot(weights_1_2[targets])
                weights_0_1[target_word] -= alpha * layer_1_delta
                weights_1_2[targets] -= alpha * np.outer(layer_2_delta, layer_1)

def similar(word_index):
    scores = Counter()
    for i in range(vocab_size):
        diff = weights_0_1[i] - weights_0_1[word_index]
        squared = diff * diff
        scores[i] = -math.sqrt(sum(squared))
    return scores.most_common(10)

def analogy(word_a, word_b, word_c):
    vec = weights_0_1[word_a] - weights_0_1[word_b] + weights_0_1[word_c]
    scores = Counter()
    for i in range(vocab_size):
        diff = weights_0_1[i] - vec
        scores[i] = -math.sqrt(sum(diff * diff))
    return scores.most_common(10)

print(similar(10))
print(similar(50))
print(analogy(20, 10, 30))

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


/tmp/ipykernel_450/285143650.py:32: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-x))


[(10, -0.0), (409, -2.856674722657127e+65), (243, -2.969996757985455e+65), (307, -3.555350764420811e+65), (6592, -3.7446523580105156e+65), (2170, -4.086728442735141e+65), (392, -4.105823253288359e+65), (1659, -4.170226538762168e+65), (1687, -5.1454421666163e+65), (138, -5.373793118323099e+65)]
[(50, -0.0), (156, -1.879485008692572e+65), (240, -1.9956705876572228e+65), (6779, -2.0077012264836735e+65), (724, -2.0776062943322705e+65), (6138, -2.132518428624788e+65), (4591, -2.1412596067516092e+65), (9575, -2.3512737587507297e+65), (661, -2.8301249132971767e+65), (6565, -2.897535526476361e+65)]
[(1619, -9.035896933553979e+65), (1401, -1.0560602226599656e+66), (2396, -1.3458411011193562e+66), (8251, -3.6060859439241274e+66), (1265, -4.6473079298736664e+66), (432, -4.686041943112935e+66), (102, -4.75079181570416e+66), (729, -5.628874892796795e+66), (2741, -6.24990930266857e+66), (1487, -8.208271489107203e+66)]


In [7]:
import numpy as np
from keras.datasets import imdb
from collections import Counter
import math

np.random.seed(42)

(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=10000)

vocab = range(10000)

input_dataset = [list(set(review)) for review in x_train]
target_dataset = list(y_train)

def sigmoid(x):
    return 1/(1+np.exp(-x))

alpha = 0.01
iterations = 10
hidden_size = 50

weights_0_1 = np.random.random((len(vocab), hidden_size))
weights_1_2 = np.random.random((hidden_size, 1))

for iter in range(iterations):
    for i in range(len(input_dataset)):
        x, y = (input_dataset[i], target_dataset[i])
        # forward propagation
        layer_1 = sigmoid(np.sum(weights_0_1[x], axis=0))
        layer_2 = sigmoid(np.dot(layer_1, weights_1_2))
        # backward propagation
        layer_2_delta = layer_2 - y
        layer_1_delta = layer_2_delta.dot(weights_1_2.T)
        # update weights
        weights_0_1[x] -= layer_1_delta * alpha
        layer_1 = layer_1.reshape(hidden_size, 1)
        layer_2_delta = layer_2_delta.reshape(1, 1)
        weights_1_2 -= np.dot(layer_1, layer_2_delta) * alpha
    print("Epoch: " + str(iter+1) + " complete")

correct, total = (0, 0)
for i in range(len(input_dataset)):
    x = input_dataset[i]
    y = target_dataset[i]
    layer_1 = sigmoid(np.sum(weights_0_1[x], axis=0))
    layer_2 = sigmoid(np.dot(layer_1, weights_1_2))
    if (np.abs(layer_2 - y) < 0.5):
        correct += 1
    total += 1
output = correct/(total)
print(output)

word_index = imdb.get_word_index()

def similar(target="fantastic"):
    target_index = word_index[target]
    scores = Counter()
    for word, index in word_index.items():
        if index >= 10000:
            continue
        raw_difference = weights_0_1[index] - (weights_0_1[target_index])
        squared_difference = raw_difference * raw_difference
        scores[word] = -math.sqrt(sum(squared_difference))
    return scores.most_common(10)

print(similar("fantastic"))
print(similar("boring"))

Epoch: 1 complete


/tmp/ipykernel_907/3185283224.py:16: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-x))


Epoch: 2 complete
Epoch: 3 complete
Epoch: 4 complete
Epoch: 5 complete
Epoch: 6 complete
Epoch: 7 complete
Epoch: 8 complete
Epoch: 9 complete
Epoch: 10 complete
0.84728
[('fantastic', -0.0), ('dancing', -2.3802214160804738), ("haven't", -2.5222911794178158), ('guess', -2.539021545083242), ('born', -2.6569373063998833), ('bring', -2.6702396909343338), ('fiction', -2.757146253822168), ('depth', -2.7736979801322765), ('45', -2.821156509583661), ('hong', -2.842111281089757)]
[('boring', -0.0), ('need', -2.9916795216776233), ('talk', -3.838756113954485), ("wasn't", -3.8538684191316914), ('worth', -3.9729983062119927), ('probably', -3.9751085741767214), ('among', -4.350937291209837), ('acting', -4.593898182575584), ('box', -4.693621277574623), ('police', -4.9334183166289085)]
